# Peers de colectores BGP — RIS y RouteViews

Itera todos los bview/rib de los colectores de una fuente dada para una fecha
y produce una lista de ASes que peerean directamente, con el total de prefijos
anunciados y el número de colectores donde ese AS aparece como peer.

Configurar `SOURCE = "ris"` o `SOURCE = "routeviews"` en la celda siguiente.

In [1]:
from pathlib import Path
import subprocess
from collections import Counter, defaultdict

SOURCE = "routeviews"   # "ris" or "routeviews"
DATE   = "20250401"

if SOURCE == "ris":
    MRT_DIR = Path(f"../mrt/{DATE}")
    GLOB    = "bview.rrc*.gz"
    def collector_name(fname):
        return fname.split(".")[1]                      # bview.rrc00.… → rrc00
else:
    MRT_DIR = Path(f"../mrt-routeviews/{DATE}")
    GLOB    = "rib.*.bz2"
    def collector_name(fname):
        return ".".join(fname.split(".")[1:-2])         # rib.route-views.amsix.….bz2 → route-views.amsix

OUTPUT = MRT_DIR.parent / f"peers-{DATE}.txt"

print(f"SOURCE  : {SOURCE}")
print(f"MRT_DIR : {MRT_DIR}")
print(f"GLOB    : {GLOB}")
print(f"OUTPUT  : {OUTPUT}")

SOURCE  : routeviews
MRT_DIR : ../mrt-routeviews/20250401
GLOB    : rib.*.bz2
OUTPUT  : ../mrt-routeviews/peers-20250401.txt


In [2]:
# field[1] == 'R'  →  RIB entry
# field[7]         →  peer ASN
peer_prefixes   = Counter()          # asn -> total prefix count across all collectors
peer_collectors = defaultdict(set)   # asn -> set of collector names

files = sorted(MRT_DIR.glob(GLOB))
print(f"Encontrados {len(files)} archivos en {MRT_DIR}\n")

for f in files:
    collector = collector_name(f.name)
    local_count = 0

    cmd = ["bgpreader", "-d", "singlefile", "-o", f"rib-file={f}"]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, text=True) as proc:
        for line in proc.stdout:
            fields = line.split("|")
            if len(fields) > 7 and fields[1] == "R":
                peer_asn = fields[7].strip()
                if peer_asn:
                    peer_prefixes[peer_asn] += 1
                    peer_collectors[peer_asn].add(collector)
                    local_count += 1
                    if local_count % 5_000_000 == 0:
                        print(f"    [{collector}] {local_count:,} entradas procesadas…")

    print(f"  {collector}: {local_count:,} entradas RIB, {sum(1 for a in peer_collectors if collector in peer_collectors[a]):,} peers")

print(f"\nDone — {len(peer_prefixes):,} ASes únicos en {len(files)} colectores")

Encontrados 10 archivos en ../mrt-routeviews/20250401



WARN: No time window specified, defaulting to all available data


    [ix-br2.gru.20250401] 5,000,000 entradas procesadas…
    [ix-br2.gru.20250401] 10,000,000 entradas procesadas…
    [ix-br2.gru.20250401] 15,000,000 entradas procesadas…
  ix-br2.gru.20250401: 19,572,581 entradas RIB, 33 peers


WARN: No time window specified, defaulting to all available data


  namex.fco.20250401: 4,833,002 entradas RIB, 11 peers


...

    [route-views.rio.20250401] 5,000,000 entradas procesadas…
    [route-views.rio.20250401] 10,000,000 entradas procesadas…
  route-views.rio.20250401: 12,068,544 entradas RIB, 17 peers


WARN: No time window specified, defaulting to all available data


    [route-views.sg.20250401] 5,000,000 entradas procesadas…
    [route-views.sg.20250401] 10,000,000 entradas procesadas…
    [route-views.sg.20250401] 15,000,000 entradas procesadas…
    [route-views.sg.20250401] 20,000,000 entradas procesadas…
  route-views.sg.20250401: 20,238,071 entradas RIB, 49 peers


WARN: No time window specified, defaulting to all available data


    [route-views2.20250401] 5,000,000 entradas procesadas…
    [route-views2.20250401] 10,000,000 entradas procesadas…
    [route-views2.20250401] 15,000,000 entradas procesadas…
    [route-views2.20250401] 20,000,000 entradas procesadas…
  route-views2.20250401: 21,506,589 entradas RIB, 26 peers

Done — 179 ASes únicos en 10 colectores


In [3]:
sorted_asns = sorted(
    peer_prefixes,
    key=lambda asn: (peer_prefixes[asn], len(peer_collectors[asn])),
    reverse=True,
)

print(f"{'ASN':>10}  {'Prefijos':>10}  {'Colectores':>10}")
print("-" * 36)
for asn in sorted_asns:
    prefixes   = peer_prefixes[asn]
    coll_count = len(peer_collectors[asn])
    print(f"{int(asn):>10}  {prefixes:>10,}  {coll_count:>10}")

lines = [f"{int(asn):08d} {peer_prefixes[asn]} {len(peer_collectors[asn])}" for asn in sorted_asns]
OUTPUT.write_text("\n".join(lines) + "\n")
print(f"\nGuardado: {OUTPUT}  ({len(lines):,} entradas)")

       ASN    Prefijos  Colectores
------------------------------------
     58511   7,169,674           3
    199524   5,953,145           4
     16552   4,776,218           4
     49544   4,770,041           4
    136557   3,385,249           2
    264409   3,160,259           2
     28329   2,452,690           1
    267613   2,397,000           2
     20253   2,389,327           2
      7713   2,384,220           1
     37271   2,367,804           1
     14630   2,365,047           1
     30844   2,363,919           2
      8220   2,356,187           2
     17350   2,354,901           1
     38880   2,251,286           2
     18106   2,214,587           2
       293   2,201,510           2
      3303   2,199,076           2
      6939   2,195,535           2
    264479   2,107,440           2
     60150   2,046,751           1
    263009   1,326,714           1
      7606   1,243,936           1
    263541   1,224,996           1
    132337   1,216,490           1
     37468   1,210